# IBM Data Analyst Capstone Project
## Module 1 — Web Scraping

**Goal:** Scrape the most popular programming languages and their average annual salaries  
from a web page using `BeautifulSoup`, then save the results to a CSV file.

In [1]:
# ── CELL 1: Import required libraries ──────────────────────────────────────────
import requests                  # to send HTTP GET requests
from bs4 import BeautifulSoup    # to parse HTML content
import pandas as pd              # to organise data and save CSV
import os                        # to build file paths

In [2]:
# ── CELL 2: Fetch the web page ─────────────────────────────────────────────────
# URL of the Wikipedia page listing programming languages by salary
url = "https://en.wikipedia.org/wiki/List_of_programming_languages_by_type"

# Send a GET request; store the response
response = requests.get(url)

# Confirm the request was successful (status 200 = OK)
print("Status code:", response.status_code)

Status code: 403


In [3]:
# ── CELL 3: Scrape programming language salary table ──────────────────────────
# Target page that contains a salary table (PYPL / Stack Overflow mirror)
salary_url = (
    "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/"
    "IBM-DA0321EN-SkillsNetwork/labs/datasets/Programming_Languages.html"
)

# Fetch and parse the page
page  = requests.get(salary_url)
soup  = BeautifulSoup(page.text, "html.parser")

# Locate the first <table> on the page
table = soup.find("table")

# Initialise empty lists to collect scraped values
languages = []
salaries  = []

# Loop over every table row (<tr>), skipping the header row
for row in table.find_all("tr")[1:]:
    cols = row.find_all("td")       # get all cells in this row
    if len(cols) >= 4:              # safety check: row must have enough columns
        lang   = cols[1].get_text(strip=True)   # column 1 = language name
        salary = cols[3].get_text(strip=True)   # column 3 = annual avg salary
        languages.append(lang)
        salaries.append(salary)

# Quick preview of scraped values
for l, s in zip(languages, salaries):
    print(f"{l:20s}  {s}")

Python                $114,383
Java                  $101,013
R                     $92,037
Javascript            $110,981
Swift                 $130,801
C++                   $113,865
C#                    $88,726
PHP                   $84,727
SQL                   $84,793
Go                    $94,082


In [4]:
# ── CELL 4: Build a DataFrame and inspect it ──────────────────────────────────
df_scraped = pd.DataFrame({
    "Language":              languages,
    "Annual_Avg_Salary_USD": salaries
})

# Show the first 10 rows
df_scraped.head(10)

,Language,Annual_Avg_Salary_USD
0,Python,"$114,383"
1,Java,"$101,013"
2,R,"$92,037"
3,Javascript,"$110,981"
4,Swift,"$130,801"
5,C++,"$113,865"
6,C#,"$88,726"
7,PHP,"$84,727"
8,SQL,"$84,793"
9,Go,"$94,082"


In [5]:
# ── CELL 5: Save scraped data to CSV ──────────────────────────────────────────
# Build output path relative to this notebook
output_path = os.path.join("..", "data", "popular_languages.csv")
os.makedirs(os.path.dirname(output_path), exist_ok=True)   # create folder if needed

df_scraped.to_csv(output_path, index=False)   # save without row numbers
print(f"Saved → {output_path}")

Saved → ..\data\popular_languages.csv
